# CWSB V2 — ConvNeXt retrain with the new close-up data (one-run Colab)

Retrains the **same ConvNeXt-Tiny classifier** (contender B recipe, `src/train.py`, seed 42)
on the original dataset **plus** the new `data/` stem close-ups added as *Unhealthy* examples.
No YOLO anywhere — the bounding-box `.txt` files are ignored; only the images are used.

**You need two zips** (both already exist at the repo root on the local machine):
- `cwsb_package.zip` (199 MB) — cleaned original images + splits + tiles + `train.py`
- `data.zip` (39 MB) — the new close-up dataset

Put both in the top level of your Google Drive (easiest for the 199 MB file), or upload when asked.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.
(If cell 1 shows no GPU, fix this first — changing it later wipes session files.)

**Honest warning, recorded here on purpose:** the new dataset contains **only infested**
close-ups (every image has WSB evidence, zero healthy close-ups). This retrain teaches the
model more about infested stems, but it may also push it further toward calling *any*
close-up infested. Judge it with cell 6, and re-test `test_new/` locally afterwards.

In [ ]:
# Cell 1 - GPU check + installs
!nvidia-smi -L || echo 'NO GPU - Runtime > Change runtime type > T4 GPU (do it NOW, it wipes files)'
%pip -q install timm
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# Cell 2 - get the two zips and unpack
from pathlib import Path

USE_DRIVE = True  # False = browser upload instead
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp /content/drive/MyDrive/cwsb_package.zip /content/drive/MyDrive/data.zip /content/
else:
    from google.colab import files
    print('upload cwsb_package.zip and data.zip'); files.upload()

!unzip -oq /content/cwsb_package.zip -d /content/cwsb   # -oq: skip the replace prompt
!unzip -oq /content/data.zip -d /content/newdata

ROOT = Path('/content/cwsb')
NEW = Path('/content/newdata')
if not (NEW / 'train').exists():  # tolerate data.zip zipped with a parent folder
    NEW = next(p for p in NEW.iterdir() if (p / 'train').exists())
assert (ROOT / 'work' / 'splits' / 'split.csv').exists(), 'cwsb_package.zip layout unexpected'
print('original data:', ROOT, '| new close-ups:', NEW)

In [ ]:
# Cell 3 - merge the new close-ups into the pipeline's data layout
# - dedupe exact copies (Roboflow shipped identical images under different .rf hashes)
# - group by base photo name and split 70/15/15 BY GROUP, seed 42
#   (Roboflow split by photo, which leaks near-copies across splits - project rule says never)
# - save into work/clean/Unhealthy/ and append rows to split.csv (label=Unhealthy)
import csv, hashlib, random, re, shutil
from collections import defaultdict
from PIL import Image

SEED = 42
CLEAN_UNH = ROOT / 'work' / 'clean' / 'Unhealthy'
SPLIT_CSV = ROOT / 'work' / 'splits' / 'split.csv'
rx = re.compile(r'^(?P<base>.+?)_(?:jpg|jpeg|png)\.rf\.[0-9a-f]{32}$', re.IGNORECASE)

seen, groups = set(), defaultdict(list)
for p in sorted(NEW.rglob('*')):
    if p.suffix.lower() not in {'.jpg', '.jpeg', '.png'} or 'labels' in p.parts:
        continue
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    if h in seen:
        continue  # exact duplicate
    seen.add(h)
    m = rx.match(p.stem)
    base = m['base'] if m else p.stem
    groups[base].append(p)

keys = sorted(groups)
random.Random(SEED).shuffle(keys)
n = len(keys)
assign = {k: ('train' if i < int(n * .70) else 'val' if i < int(n * .85) else 'test')
          for i, k in enumerate(keys)}

rows, counts = [], defaultdict(int)
for base, paths in groups.items():
    for j, p in enumerate(sorted(paths)):
        name = f'new_{base}_{j}.jpg'
        Image.open(p).convert('RGB').save(CLEAN_UNH / name, 'JPEG', quality=90)
        rows.append({'file': f'Unhealthy/{name}', 'label': 'Unhealthy',
                     'group': f'closeup_{base}', 'split': assign[base]})
        counts[assign[base]] += 1

with open(SPLIT_CSV, 'a', newline='') as f:
    csv.DictWriter(f, fieldnames=['file', 'label', 'group', 'split']).writerows(rows)

print(f'{len(seen)} unique new images from {n} base photos -> {dict(counts)}')
with open(SPLIT_CSV) as f:
    allrows = list(csv.DictReader(f))
tally = defaultdict(int)
for r in allrows:
    tally[(r['split'], r['label'])] += 1
print('combined split.csv:', dict(sorted(tally.items())))

In [ ]:
# Cell 4 - leakage guard, same rule as make_split.py: abort if any group is in two splits
gs = defaultdict(set)
for r in allrows:
    gs[r['group']].add(r['split'])
bad = {g: s for g, s in gs.items() if len(s) > 1}
assert not bad, f'LEAKAGE: {bad}'
print(f'OK - {len(gs)} groups, each in exactly one split')

In [ ]:
# Cell 5 - train contender B on the combined data (the project's own train.py, unmodified)
# ~30-60 min on a T4. Use --mode both to also refit contender A.
!cd /content/cwsb && python src/train.py --mode B

In [ ]:
# Cell 6 - evaluate the new checkpoint on the held-out TEST split, old vs new photos separately
import sys, numpy as np, timm
sys.path.insert(0, '/content/cwsb/src')
import train as T

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ck = torch.load(ROOT / 'models' / 'v1' / 'contender_B.pt', map_location=device, weights_only=False)
model = timm.create_model(ck['backbone'], pretrained=False, num_classes=2).to(device)
model.load_state_dict(ck['state_dict'])
print(f"checkpoint: epoch {ck['epoch']}  val PR-AUC {ck['val_pr_auc']:.4f}")

T_LOW, T_HIGH = 0.50, 0.90  # v1 thresholds; re-derive for v2 from HEALTHY scores, never infected quantiles
test_rows = T.load_rows('test')
ys, ps = T.scores(model, test_rows, device)

def report(tag, idx):
    y, p = ys[idx], ps[idx]
    v = np.where(p >= T_HIGH, 'wsb_likely', np.where(p < T_LOW, 'no_signs', 'inconclusive'))
    caught = int(((y == 1) & (v == 'wsb_likely')).sum()); missed = int(((y == 1) & (v == 'no_signs')).sum())
    false_alarm = int(((y == 0) & (v == 'wsb_likely')).sum()); cleared = int(((y == 0) & (v == 'no_signs')).sum())
    inc = int((v == 'inconclusive').sum())
    print(f'{tag:28s} n={len(y):3d}  caught {caught:3d}  missed {missed:2d}  false-alarms {false_alarm:2d}  cleared {cleared:2d}  inconclusive {inc:2d}')

is_new = np.array([r['group'].startswith('closeup_') for r in test_rows])
report('ALL test photos', np.arange(len(ys)))
report('original whole-plant test', np.where(~is_new)[0])
report('new close-up test', np.where(is_new)[0])
print('\nhealthy-photo score range (sets T_low for v2): '
      f'{ps[ys == 0].min():.3f} - {ps[ys == 0].max():.3f}' if (ys == 0).any() else 'no healthy in test')

In [ ]:
# Cell 7 - package and download as V2 (keep v1 untouched locally!)
!mkdir -p /content/out
!cp /content/cwsb/models/v1/contender_B.pt /content/out/contender_B_v2.pt
!cp /content/cwsb/models/v1/training_result.json /content/out/training_result_v2.json 2>/dev/null || true
!cp /content/cwsb/work/splits/split.csv /content/out/split_v2.csv
!cd /content/out && zip -q /content/cwsb_v2_model.zip .  -r
from google.colab import files
files.download('/content/cwsb_v2_model.zip')

## After downloading — on the local machine

1. Unzip **flat** into `models/v2/` (remember the nesting pitfall). **Do not overwrite `models/v1/`** —
   v1 is the currently-measured model; v2 is unproven until tested.
2. Copy `split_v2.csv` to `work/splits/` under that name (keep the original `split.csv` too).
3. Re-run the local checks against v2: `test2/` (the broad test set) and especially `test_new/`
   (the 19 healthy close-ups). **If v2 still flags those healthy close-ups, the new data did not fix
   the real problem** — that needs healthy close-ups / look-alike negatives, not more infested ones.
4. Re-derive thresholds from the **healthy** validation score distribution (never from infected
   quantiles — that was the documented v1 bug).
5. Log the run in `charan_working.md` and update `PROJECT_STATE.md`.